# End-to-End Provider Directory Update Pipeline Demo

This notebook demonstrates the full AI-powered pipeline on sample healthcare provider data.

## Step 1: Setup & Initialize Database

In [ ]:
import sys
import os
sys.path.insert(0, "..")
os.chdir("..")

from pipeline.db.store import get_db, init_db
from pipeline.db.seed import seed_db
from pipeline.orchestrator import run_pipeline, run_batch
from pipeline.agents.staleness import detect_stale

conn = get_db()
init_db(conn)
seed_db(conn)
print("DB initialized with sample providers.")

## Step 2: Detect Stale Records

In [ ]:
stale = detect_stale(conn, days=90)
print(f"Found {len(stale)} stale records:\n")
for r in stale:
    print(f"  {r['provider_id']} | {r['provider_name']} | last verified: {r['last_verified_date']}")

## Step 3: Run Pipeline on One Record (Mocked for Offline Demo)

In [ ]:
from unittest.mock import patch

record = stale[0]
print(f"Processing: {record['provider_id']} — {record['provider_name']}\n")

mock_nppes = {
    "provider_name": record["provider_name"],
    "address": "250 Health Park Dr, Fort Myers, FL 33908",
    "phone": "239-555-9000",
    "specialty": record["specialty"],
    "active": True,
    "practice_name": "",
}
mock_cms = {
    "provider_name": record["provider_name"],
    "specialty": record["specialty"].upper(),
    "practice_name": "Fort Myers Medical Group",
    "active": True,
    "address": "250 Health Park Dr, Fort Myers, FL 33908",
    "phone": "239-555-9000",
}

with patch("pipeline.agents.fetch.fetch_nppes", return_value=mock_nppes), \
     patch("pipeline.agents.fetch.fetch_cms", return_value=mock_cms), \
     patch("pipeline.agents.fetch.fetch_florida_board", return_value=None), \
     patch("pipeline.agents.fetch.fetch_website", return_value=None):
    result = run_pipeline(record, conn)

print(f"Action: {result['recommended_action']}")
print(f"Confidence: {result['overall_confidence']}")
print(f"Reason: {result['reason']}\n")
for diff in result["diffs"]:
    print(f"  {diff['field']}: {diff['old_value']!r} → {diff['new_value']!r} "
          f"(score={diff['confidence_score']}, sources={diff['supporting_sources']})")

## Step 4: View Audit Log

In [ ]:
import sqlite3
rows = conn.execute(
    "SELECT provider_id, field, old_value, new_value, confidence_score, action, reason "
    "FROM audit_log ORDER BY id DESC LIMIT 10"
).fetchall()
for row in rows:
    print(dict(row))

## Cleanup

In [ ]:
conn.close()
print("Demo complete.")